In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QUICK-PDE: Fungsi Qiskit oleh ColibriTD
> **Note:** Fungsi Qiskit adalah ciri eksperimental yang tersedia untuk pengguna Pelan Premium IBM Quantum&reg;, Pelan Flex, dan Pelan On-Prem (melalui API Platform IBM Quantum). Ia dalam status keluaran pratonton dan boleh berubah.
## Gambaran Keseluruhan
Penyelesai Persamaan Pembezaan Separa (PDE) yang dibentangkan di sini adalah sebahagian daripada platform Quantum Innovative Computing Kit (QUICK) kami (QUICK-PDE), dan dipaketkan sebagai Fungsi Qiskit. Dengan fungsi QUICK-PDE, anda boleh menyelesaikan persamaan pembezaan separa khusus domain pada QPU IBM Quantum. Fungsi ini berdasarkan algoritma yang diterangkan dalam [kertas penerangan H-DES ColibriTD.](https://arxiv.org/abs/2410.01130) Algoritma ini mampu menyelesaikan masalah berbilang fizik yang kompleks, bermula dengan Dinamik Cecair Pengiraan (CFD) dan Ubah Bentuk Bahan (MD), serta kes penggunaan lain yang akan hadir tidak lama lagi.

Untuk menangani persamaan pembezaan, penyelesaian cubaan dikodkan sebagai gabungan linear fungsi ortogon (biasanya polinomial Chebyshev, dan lebih khususnya $2^n$ daripadanya di mana $n$ ialah bilangan qubit yang mengekod fungsi anda), diparameter oleh sudut Litar Kuantum Boleh Ubah (VQC). Ansatz menjana keadaan yang mengekod fungsi, yang dinilai oleh boleh cerap yang gabungannya membolehkan penilaian fungsi pada semua titik. Anda kemudian boleh menilai fungsi kerugian di mana persamaan pembezaan dikodkan, dan melaraskan sudut dalam gelung hibrid, seperti yang ditunjukkan berikut. Penyelesaian cubaan semakin menghampiri penyelesaian sebenar sehingga anda mencapai hasil yang memuaskan.

![Aliran kerja fungsi QUICK-PDE](../docs/images/guides/colibritd-equation-solver/diagram.svg)

Selain gelung hibrid ini, anda juga boleh menyambungkan pengoptimum yang berbeza secara bersiri. Ini berguna apabila anda mahu pengoptimum global mencari set sudut yang baik, kemudian pengoptimum yang lebih halus untuk mengikuti kecerunan ke set sudut jiran terbaik. Dalam kes dinamik cecair pengiraan (CFD), urutan pengoptimuman lalai menghasilkan keputusan terbaik — tetapi dalam kes ubah bentuk bahan (MD), walaupun lalai memberikan keputusan yang baik, anda boleh mengkonfigurasinya lebih lanjut untuk manfaat khusus masalah.

Perhatikan bahawa bagi setiap pemboleh ubah fungsi, kami menentukan bilangan qubit (yang boleh anda ubah). Dengan menindih 10 litar yang sama dan menilai 10 boleh cerap yang sama pada qubit berbeza sepanjang satu litar besar, anda boleh mengurangkan hingar dalam proses pengoptimuman CMA, bergantung pada kaedah pembelajaran hingar, dan mengurangkan bilangan tembakan yang diperlukan dengan ketara.
## Input/output
### Dinamik Cecair Pengiraan
Persamaan Burgers' tak likat, memodelkan cecair tidak likat yang mengalir seperti berikut:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$ mewakili medan kelajuan cecair. Kes penggunaan ini mempunyai syarat sempadan temporal: anda boleh memilih keadaan awal dan kemudian membenarkan sistem berehat. Buat masa ini, satu-satunya syarat awal yang diterima ialah fungsi linear: $ax + b$.

Hujah untuk persamaan pembezaan CFD berada pada grid tetap, seperti berikut:

- $t$ berada antara 0 dan 0.95 dengan 30 titik sampel. $x$ berada antara 0 dan 0.95 dengan saiz langkah 0.2375.

### Ubah Bentuk Bahan
Kes penggunaan ini memberi tumpuan pada ubah bentuk hipoelastik dengan ujian tegangan satu dimensi, di mana sebatang rod yang ditetapkan dalam ruang ditarik pada hujung satunya. Kami menerangkan masalah seperti berikut:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$ mewakili modulus pukal bahan yang diregangkan, $n$ eksponen hukum kuasa, $b$ daya per unit jisim, $\epsilon_0$ had tegasan berkadar, $\sigma_0$ had terikan berkadar, $u$ fungsi tegasan, dan $\sigma$ fungsi terikan.

Rod yang dipertimbangkan mempunyai panjang padu. Kes penggunaan ini mempunyai syarat sempadan untuk tegasan permukaan $t$, atau jumlah kerja yang diperlukan untuk meregangkan rod.

Hujah untuk persamaan pembezaan MD berada pada grid tetap, seperti berikut:

- $x$ berada antara 0 dan 1 dengan saiz langkah 0.04.
### Input
Untuk menjalankan Fungsi Qiskit QUICK-PDE, anda boleh melaraskan parameter berikut:

| Nama              | Jenis                                                | Penerangan                                                                                                                                                                                                                                                                      | Khusus kes penggunaan | Contoh                   |
| ----------------- | ---------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | --------------------- | ------------------------ |
| use_case          | `Literal["MD", "CFD"]`                               | Togol untuk memilih sistem persamaan pembezaan yang hendak diselesaikan                                                                                                                                                                                                         | Tidak                 | `"CFD"`                  |
| parameters        | `dict[str, Any]`                                     | Parameter persamaan pembezaan (lihat jadual seterusnya untuk maklumat lanjut)                                                                                                                                                                                                   | Tidak                 | `{"a": 1.0, "b": 1.0}`   |
| nb_qubits         | `Optional[dict[str, dict[str, int]]]`                | Bilangan qubit per fungsi dan per pemboleh ubah. Nilai dioptimumkan dipilih oleh fungsi, tetapi jika anda ingin mencuba gabungan yang lebih baik, anda boleh mengatasi nilai lalai                                                                                              | Tidak                 | `{"u": {"x": 1, "t":3}}` |
| depth             | `Optional[dict[str, int]]`                           | Kedalaman ansatz per fungsi. Nilai dioptimumkan dipilih oleh fungsi, tetapi jika anda ingin mencuba gabungan yang lebih baik, anda boleh mengatasi nilai lalai                                                                                                                  | Tidak                 | `{"u": 4}`               |
| optimizer         | `Optional[list[str]]`                                | Pengoptimum yang akan digunakan, sama ada "CMAES" daripada [perpustakaan python `cma`](https://github.com/CMA-ES/pycma) atau salah satu [pengoptimum scipy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)                                  | MD                    | `"SLSQP"`                |
| shots             | `Optional[list[int]]`                                | Bilangan tembakan yang digunakan untuk menjalankan setiap litar. Memandangkan beberapa langkah pengoptimuman diperlukan, panjang senarai mestilah sama dengan bilangan pengoptimum yang digunakan (4 untuk CFD). Lalai kepada `[50_000] * nb_optimizers` untuk MD dan `[5_00, 2_000, 5_000, 10_000]` untuk CFD | Tidak | `[15_000, 30_000]`       |
| optimizer_options | `Optional[dict[str, Any]]`                           | Pilihan untuk dihantar kepada pengoptimum. Butiran input ini bergantung pada pengoptimum yang digunakan; untuk maklumat lanjut, rujuk dokumentasi pengoptimum yang digunakan                                                                                                    | MD                    | `{"maxiter": 50 }`       |
| initialization    | `Optional[Literal["RANDOM", "PHYSICALLY_INFORMED"]]` | Sama ada untuk bermula dengan sudut rawak atau sudut yang dipilih secara bijak. Perhatikan bahawa sudut yang dipilih secara bijak mungkin tidak berfungsi dalam 100% kes. Lalai kepada `"PHYSICALLY_INFORMED"`.                                                                 | Tidak                 | `"RANDOM"`               |
| backend_name      | `Optional[str]`                                      | Nama backend yang hendak digunakan.                                                                                                                                                                                                                                             | Tidak                 | `"ibm_torino"`           |
| mode              | `Optional[Literal["job", "session", "batch"]]`        | Mod pelaksanaan yang hendak digunakan. Lalai kepada `"job"`.                                                                                                                                                                                                                   | Tidak                 | `"job"`                  |

Parameter persamaan pembezaan (parameter fizikal dan syarat sempadan) hendaklah mengikuti format yang diberikan:

| Kes penggunaan | Kunci       | Jenis nilai | Penerangan                                  | Contoh  |
| -------------- | ----------- | ----------- | ------------------------------------------- | ------- |
| CFD            | `a`         | `float`     | Pekali nilai awal $u$                       | `1.0`   |
| CFD            | `b`         | `float`     | Ofset nilai awal $u$                        | `1.0`   |
| MD             | `t`         | `float`     | tegasan permukaan                           | `12.0`  |
| MD             | `K`         | `float`     | modulus pukal                               | `100.0` |
| MD             | `n`         | `int`       | hukum kuasa                                 | `4.0`   |
| MD             | `b`         | `float`     | daya per unit jisim                         | `10.0`  |
| MD             | `epsilon_0` | `float`     | had tegasan berkadar                        | `0.1`   |
| MD             | `sigma_0`   | `float`     | had terikan berkadar                        | `5.0`   |
### Output
Output ialah kamus dengan senarai titik sampel, serta nilai fungsi pada setiap titik tersebut:

In [ ]:
from numpy import array

In [ ]:
solution = {
    "functions": {
        "u": array(
            [
                [0.01, 0.1, 1],
                [0.02, 0.2, 2],
                [0.03, 0.3, 3],
                [0.04, 0.4, 4],
            ]
        ),
    },
    "samples": {
        "t": array([0.1, 0.2, 0.3, 0.4]),
        "x": array([0.5, 0.6, 0.7]),
    },
}

Bentuk tatasusunan penyelesaian bergantung pada sampel pemboleh ubah:

In [ ]:
assert len(solution["functions"]["u"].shape) == len(solution["samples"])
for col_size, samples in zip(
    solution["functions"]["u"].shape, solution["samples"].values()
):
    assert col_size == len(samples)

Pemetaan antara titik sampel pemboleh ubah fungsi dan dimensi tatasusunan penyelesaian dilakukan mengikut urutan alfanumerik nama pemboleh ubah. Contohnya, jika pemboleh ubah adalah `"t"` dan `"x"`, satu baris `solution["functions"]["u"]` mewakili nilai penyelesaian untuk `"t"` yang tetap, dan satu lajur `solution["functions"]["u"]` mewakili nilai penyelesaian untuk `"x"` yang tetap.

Berikut ialah contoh cara mendapatkan nilai fungsi untuk set koordinat tertentu:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

## Penanda Aras

Jadual berikut memaparkan statistik pelbagai jalankan fungsi kami.

| Contoh                         | Bilangan qubit | Pemulaan             | Ralat     | Jumlah masa (min) | Penggunaan masa jalan (min) |
| ------------------------------ | -------------- | -------------------- | --------- | ----------------- | --------------------------- |
| Persamaan Burgers' tak likat   | 50             | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66                | 25                          |
| Ujian tegangan 1D hipoelastik  | 18             | `RANDOM`             | $10^{-2}$ | 123               | 100                         |

## Mulakan

Isi [borang untuk meminta akses kepada fungsi QUICK-PDE.](https://forms.cloud.microsoft/e/3Wi9cbjQPK) Kemudian, dengan mengandaikan anda sudah [menyimpan akaun anda](/guides/functions#install-qiskit-functions-catalog-client) ke persekitaran tempatan anda, pilih fungsi seperti berikut:

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

quick = catalog.load("colibritd/quick-pde")

## Contoh

Untuk bermula, cuba salah satu contoh berikut:

### Dinamik Cecair Pengiraan (CFD)

Apabila syarat awal ditetapkan kepada $u(0,x) = x$, hasilnya adalah seperti berikut:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(use_case="cfd", physical_parameters={"a": 1.0, "b": 0.0})

Semak [status](/guides/functions#check-job-status) atau dapatkan semula [keputusan](/guides/functions#retrieve-results) beban kerja Fungsi Qiskit anda seperti berikut:

In [ ]:
print(job.status())
solution = job.result()

'QUEUED'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

_ = plt.figure()
ax = plt.axes(projection="3d")

# plot the solution using the 3d plotting capabilities of pyplot
t, x = np.meshgrid(solution["samples"]["t"], solution["samples"]["x"])
ax.plot_surface(
    t,
    x,
    solution["functions"]["u"],
    edgecolor="royalblue",
    lw=0.25,
    rstride=26,
    cstride=26,
    alpha=0.3,
)
ax.scatter(t, x, solution["functions"]["u"], marker=".")
ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif" alt="Output of the previous code cell" />

### Material Deformation

The material deformation use case requires the physical parameters of your material and the applied force, as follows:

In [ ]:
import matplotlib.pyplot as plt

# select the properties of your material
job = quick.run(
    use_case="md",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### Ubah Bentuk Bahan
Kes penggunaan ubah bentuk bahan memerlukan parameter fizikal bahan anda dan daya yang dikenakan, seperti berikut:

In [ ]:
job = quick.run(use_case="mdf", physical_params={})

print(job.error_message())


# or write a wrapper around it for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

{"error": ["qiskit.exceptions.QiskitError: 'Unknown argument \"physical_params\", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'\n"]}
___
qiskit.exceptions.QiskitError: 'Unknown argument "physical_params", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'



![Output of the previous code cell](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

## Dapatkan Mesej Ralat

Jika status beban kerja anda ialah `ERROR`, gunakan `job.error_message()` untuk mendapatkan mesej ralat bagi membantu nyahpepijat, seperti berikut: